# routes.handlers

> Response builder functions for virtual collection navigation (Tier 1 API).

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Tuple

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.windowing import navigate, clamp_window_start
from cjm_fasthtml_virtual_collection.components.table import render_table_rows
from cjm_fasthtml_virtual_collection.components.footer import render_footer

## Response Builders

In [ ]:
#| export
def build_nav_response(
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (already mutated)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
) -> Tuple:  # OOB elements (rows + footer)
    """Build OOB response for navigation: rows + footer."""
    rows_oob = render_table_rows(items, config, state, ids, render_cell)
    rows_oob.attrs["hx-swap-oob"] = "innerHTML"

    footer_oob = render_footer(state, ids, oob=True)

    return (rows_oob, footer_oob)

## Navigation Handlers (Tier 1)

In [ ]:
#| export
def handle_navigate(
    direction: str,                         # 'up', 'down', 'page_up', 'page_down', 'first', 'last'
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
) -> Tuple:  # OOB elements (rows + footer)
    """Navigate in a direction. Mutates state.window_start in place."""
    state.window_start = navigate(
        state.window_start, direction, state.visible_rows, state.total_items
    )
    return build_nav_response(items, state, config, ids, render_cell)

In [ ]:
#| export
def handle_navigate_to_index(
    target_index: int,                      # Target window_start
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
) -> Tuple:  # OOB elements (rows + footer)
    """Navigate to a specific index. Mutates state.window_start in place."""
    state.window_start = clamp_window_start(
        target_index, state.visible_rows, state.total_items
    )
    return build_nav_response(items, state, config, ids, render_cell)

In [ ]:
#| export
def handle_update_viewport(
    visible_rows: int,                      # New visible row count
    items: list,                            # Full item list
    state: VirtualCollectionState,          # Current state (mutated in place)
    config: VirtualCollectionConfig,        # Collection config
    ids: VirtualCollectionHtmlIds,          # HTML IDs
    render_cell: Callable,                  # Consumer cell render callback
    is_auto: bool = True,                   # Whether from auto-fit
) -> Tuple:  # OOB elements (rows + footer)
    """Update viewport with new row count. Mutates state in place."""
    state.visible_rows = max(1, visible_rows)
    state.is_auto_mode = is_auto
    # Re-clamp window_start in case new visible_rows changes the valid range
    state.window_start = clamp_window_start(
        state.window_start, state.visible_rows, state.total_items
    )
    return build_nav_response(items, state, config, ids, render_cell)

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import to_xml, Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds

@dataclass
class Item:
    name: str

test_items = [Item(f"file_{i}.txt") for i in range(50)]
columns = (ColumnDef(key="name", header="Name", width="1fr"),)
config = VirtualCollectionConfig(prefix="th", columns=columns)
ids = VirtualCollectionHtmlIds(prefix="th")

def test_render_cell(item, ctx):
    return Span(item.name)

In [ ]:
# Test handle_navigate down
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0)
result = handle_navigate("down", test_items, state, config, ids, test_render_cell)
assert state.window_start == 1
assert len(result) == 2  # rows + footer

rows_html = to_xml(result[0])
assert 'hx-swap-oob="innerHTML"' in rows_html
assert 'file_1.txt' in rows_html  # starts at row 1 now

footer_html = to_xml(result[1])
assert 'hx-swap-oob="outerHTML"' in footer_html
assert 'Showing 2-6 of 50 items' in footer_html
print("handle_navigate down test passed")

handle_navigate down test passed


In [ ]:
# Test handle_navigate clamping
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0)
result = handle_navigate("up", test_items, state, config, ids, test_render_cell)
assert state.window_start == 0  # clamped at 0
print("handle_navigate clamping test passed")

handle_navigate clamping test passed


In [ ]:
# Test handle_navigate_to_index
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=0)
result = handle_navigate_to_index(25, test_items, state, config, ids, test_render_cell)
assert state.window_start == 25

rows_html = to_xml(result[0])
assert 'file_25.txt' in rows_html
print("handle_navigate_to_index test passed")

handle_navigate_to_index test passed


In [ ]:
# Test handle_update_viewport
state = VirtualCollectionState(total_items=50, visible_rows=5, window_start=46)
result = handle_update_viewport(10, test_items, state, config, ids, test_render_cell)
assert state.visible_rows == 10
assert state.window_start == 40  # re-clamped: 50 - 10 = 40
print("handle_update_viewport test passed")

handle_update_viewport test passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()